# Predicting E-Commerce Purchase Completion from Online Session Behaviour: A Comparative Machine Learning Analysis Using Python

Dataset: Sakar, C. and Kastro, Y. (2018), UCI Machine Learning Repository, DOI 10.24432/C5F88Q.

This notebook documents the reproducible workflow used in the dissertation. The complete runnable version is also supplied as `Python_Analysis_FINAL.py`.

**Runtime note:** The full five-fold validation uses 500-tree random forests and may take several minutes depending on the computer.


In [ ]:
import warnings, math
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu, chi2_contingency
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score, confusion_matrix, roc_curve
from sklearn.inspection import permutation_importance
RANDOM_STATE=42

## 1. Load and inspect the exact secondary dataset

In [ ]:
df=pd.read_csv('online_shoppers_intention.csv')
for col in ['Weekend','Revenue']:
    if df[col].dtype==object:
        df[col]=df[col].astype(str).str.upper().map({'TRUE':True,'FALSE':False})
print('Shape:',df.shape)
print('Missing values:',int(df.isna().sum().sum()))
print('Exact repeated profiles:',int(df.duplicated().sum()))
df['Revenue'].value_counts()

**Verified output:** 12,330 rows and 18 variables; no missing values; 125 identical recorded profiles retained because no unique session identifier is available; 10,422 non-purchase sessions and 1,908 purchase sessions.

## 2. Descriptive and association analysis

In [ ]:
key_vars=['ProductRelated','ProductRelated_Duration','BounceRates','ExitRates','PageValues','SpecialDay']
print(df.groupby('Revenue')[key_vars].mean().T)
corr_df=df.copy(); corr_df['Revenue_binary']=corr_df['Revenue'].astype(int)
print(corr_df.select_dtypes(include=np.number).corr()['Revenue_binary'].drop('Revenue_binary').sort_values(key=lambda s:s.abs(),ascending=False))

In [ ]:
numeric_tests=[]
for var in key_vars:
    no=df.loc[~df['Revenue'],var].values
    yes=df.loc[df['Revenue'],var].values
    U_no,p=mannwhitneyu(no,yes,alternative='two-sided')
    U_yes=len(no)*len(yes)-U_no
    U_reported=min(U_no,U_yes)
    rbc=2*U_yes/(len(no)*len(yes))-1
    numeric_tests.append([var,U_reported,p,rbc])
numeric_tests_df=pd.DataFrame(numeric_tests,columns=['Variable','Mann_Whitney_U','p_value','Rank_biserial_effect'])
numeric_tests_df

All six selected numerical variables were significant at p<0.001. The largest rank-biserial effect was for PageValues (0.726588), followed by ExitRates (-0.406204), ProductRelated_Duration (0.345999) and ProductRelated (0.316865).

In [ ]:
categorical_tests=[]
for var in ['VisitorType','Month','Weekend']:
    table=pd.crosstab(df[var],df['Revenue'])
    chi2,p,dof,_=chi2_contingency(table)
    n=table.values.sum(); cramer_v=math.sqrt(chi2/(n*min(table.shape[0]-1,table.shape[1]-1)))
    categorical_tests.append([var,chi2,dof,p,cramer_v])
pd.DataFrame(categorical_tests,columns=['Variable','Chi_square','df','p_value','Cramers_V'])

## 3. Preprocessing, train-test split and models

In [ ]:
X=df.drop(columns='Revenue').copy(); y=df['Revenue'].astype(int)
cat_cols=['Month','OperatingSystems','Browser','Region','TrafficType','VisitorType','Weekend']
num_cols=[c for c in X.columns if c not in cat_cols]
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.20,stratify=y,random_state=RANDOM_STATE)
pre_scaled=ColumnTransformer([('num',StandardScaler(),num_cols),('cat',OneHotEncoder(handle_unknown='ignore'),cat_cols)])
pre_unscaled=ColumnTransformer([('num','passthrough',num_cols),('cat',OneHotEncoder(handle_unknown='ignore'),cat_cols)])
models={
'Logistic Regression':Pipeline([('preprocess',pre_scaled),('model',LogisticRegression(max_iter=2000,random_state=RANDOM_STATE))]),
'Decision Tree':Pipeline([('preprocess',pre_unscaled),('model',DecisionTreeClassifier(random_state=RANDOM_STATE,min_samples_leaf=5))]),
'Random Forest':Pipeline([('preprocess',pre_unscaled),('model',RandomForestClassifier(n_estimators=500,random_state=RANDOM_STATE,n_jobs=-1,min_samples_leaf=2))])}

## 4. Held-out performance

In [ ]:
rows=[]; predictions={}; probabilities={}
for name,model in models.items():
    model.fit(X_train,y_train)
    pred=model.predict(X_test); prob=model.predict_proba(X_test)[:,1]
    predictions[name]=pred; probabilities[name]=prob
    rows.append([name,accuracy_score(y_test,pred),precision_score(y_test,pred),recall_score(y_test,pred),f1_score(y_test,pred),roc_auc_score(y_test,prob),average_precision_score(y_test,prob)])
results=pd.DataFrame(rows,columns=['Model','Accuracy','Precision','Recall','F1_score','ROC_AUC','PR_AUC'])
results

**Verified held-out results:** Logistic Regression: accuracy 0.880779, F1 0.482394, ROC-AUC 0.882787, PR-AUC 0.618895. Decision Tree: accuracy 0.873479, F1 0.569061, ROC-AUC 0.821653, PR-AUC 0.519925. Random Forest: accuracy 0.897810, F1 0.594855, ROC-AUC 0.918458, PR-AUC 0.730169.

### Majority-class baseline

In [ ]:
dummy=DummyClassifier(strategy='most_frequent').fit(X_train,y_train)
dummy_pred=dummy.predict(X_test)
dummy_prob=dummy.predict_proba(X_test)[:,1]
print('Majority-class accuracy:',accuracy_score(y_test,dummy_pred))
print('Positive-class prevalence / PR-AUC baseline:',average_precision_score(y_test,dummy_prob))

## 5. Five-fold stratified cross-validation

In [ ]:
cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=RANDOM_STATE)
scoring={'Accuracy':'accuracy','Precision':'precision','Recall':'recall','F1_score':'f1','ROC_AUC':'roc_auc','PR_AUC':'average_precision'}
cv_rows=[]
for name,model in models.items():
    scores=cross_validate(model,X,y,cv=cv,scoring=scoring,n_jobs=-1)
    row={'Model':name}
    for metric in scoring:
        vals=scores['test_'+metric]
        row[metric+'_mean']=vals.mean(); row[metric+'_SD']=vals.std(ddof=1)
    cv_rows.append(row)
pd.DataFrame(cv_rows)

**Verified cross-validation:** Random Forest mean ROC-AUC 0.928179 (SD 0.005411), mean PR-AUC 0.746005 (SD 0.015320), and mean F1 0.619838 (SD 0.025159).

## 6. Feature importance and sensitivity analysis

In [ ]:
rf=models['Random Forest']
pre=rf.named_steps['preprocess']; rf_model=rf.named_steps['model']
feature_names=pre.get_feature_names_out()
importance=pd.DataFrame({'Feature':feature_names,'Importance':rf_model.feature_importances_}).sort_values('Importance',ascending=False)
importance.head(15)

### Permutation importance on the held-out test set

In [ ]:
perm=permutation_importance(rf,X_test,y_test,n_repeats=5,scoring='roc_auc',random_state=RANDOM_STATE,n_jobs=-1,max_samples=0.75)
perm_importance=pd.DataFrame({'Feature':X_test.columns,'Mean_ROC_AUC_Decrease':perm.importances_mean,'SD':perm.importances_std}).sort_values('Mean_ROC_AUC_Decrease',ascending=False)
perm_importance.head(15)

PageValues was the dominant impurity-based feature with importance 0.399578. A held-out permutation-importance check also ranked PageValues first by a wide margin.

In [ ]:
X_reduced=df.drop(columns=['Revenue','PageValues']).copy()
num_reduced=[c for c in X_reduced.columns if c not in cat_cols]
Xr_train,Xr_test,yr_train,yr_test=train_test_split(X_reduced,y,test_size=.20,stratify=y,random_state=RANDOM_STATE)
pre_r_scaled=ColumnTransformer([('num',StandardScaler(),num_reduced),('cat',OneHotEncoder(handle_unknown='ignore'),cat_cols)])
pre_r_unscaled=ColumnTransformer([('num','passthrough',num_reduced),('cat',OneHotEncoder(handle_unknown='ignore'),cat_cols)])
reduced_models={
'Logistic Regression':Pipeline([('preprocess',pre_r_scaled),('model',LogisticRegression(max_iter=2000,random_state=RANDOM_STATE))]),
'Decision Tree':Pipeline([('preprocess',pre_r_unscaled),('model',DecisionTreeClassifier(random_state=RANDOM_STATE,min_samples_leaf=5))]),
'Random Forest':Pipeline([('preprocess',pre_r_unscaled),('model',RandomForestClassifier(n_estimators=500,random_state=RANDOM_STATE,n_jobs=-1,min_samples_leaf=2))])}
reduced=[]
for name,model in reduced_models.items():
    model.fit(Xr_train,yr_train); pred=model.predict(Xr_test); prob=model.predict_proba(Xr_test)[:,1]
    reduced.append([name,accuracy_score(yr_test,pred),precision_score(yr_test,pred,zero_division=0),recall_score(yr_test,pred,zero_division=0),f1_score(yr_test,pred,zero_division=0),roc_auc_score(yr_test,prob),average_precision_score(yr_test,prob)])
pd.DataFrame(reduced,columns=results.columns)

**Verified sensitivity result:** without PageValues, Random Forest ROC-AUC fell to 0.760603 and PR-AUC to 0.356611, demonstrating substantial dependence on this feature.

## 7. Reproducibility
The exact CSV, Python script, requirements file and Excel output workbook are supplied in the technical package. `random_state=42` is used for the train-test split and stochastic models.